# Pedestrian Trajectory Analysis — Carrefour Melen, Yaounde

Interactive exploration of the Yaounde tracking results and comparison with ETH/UCY benchmarks.

## Dataset overview

| File | Duration | Vantage point |
|---|---|---|
| `Record_1.mp4` | 10m01s | Street level |
| `Record_2.mp4` | 5m00s  | Street level |
| `Record 3.mp4` | 5m00s  | 3rd-floor balcony |
| `Record 4.mp4` | 5m00s  | 3rd-floor balcony |
| `Record 5.mp4` | 5m00s  | 3rd-floor balcony |
| `Record 6.mp4` | 5m00s  | 3rd-floor balcony |

> **Note:** Balcony recordings (3-6) are used for the main trajectory analysis and ETH/UCY comparison,
> as the overhead view minimises occlusion and perspective distortion.

## Workflow
1. Setup & imports
2. Load trajectories (per recording + merged groups)
3. Inspect basic statistics per recording and per group
4. Visualise heatmaps & trajectories
5. Load ETH/UCY data
6. Side-by-side comparison (balcony group vs ETH/UCY)
7. Per-recording breakdown

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from extract_traj import compute_speeds
from visualize import plot_heatmap, plot_trajectories, plot_speed_distribution
from compare_eth_ucy import load_eth_ucy, per_pedestrian_stats, plot_comparison

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# ---------------------------------------------------------------
# Recording registry (mirrors run_all.py)
# ---------------------------------------------------------------
RECORDINGS = [
    ("record_1",  "../data/Record_1.mp4",  "ground"),
    ("record_2",  "../data/Record_2.mp4",  "ground"),
    ("record_3",  "../data/Record 3.mp4",  "balcony"),
    ("record_4",  "../data/Record 4.mp4",  "balcony"),
    ("record_5",  "../data/Record 5.mp4",  "balcony"),
    ("record_6",  "../data/Record 6.mp4",  "balcony"),
]

TRAJ_DIR   = Path("../data/trajectories")
RESULTS_DIR = Path("../results")
ETH_DIR    = Path("../data/eth_ucy")

# Calibration constants (adjust after ground-truth measurement)
FPS      = 30.0
PX_PER_M = {"ground": 50.0, "balcony": 30.0}


## 2. Load Trajectories

In [ ]:
# Load individual recordings
dfs = {}
for name, _, group in RECORDINGS:
    csv_path = TRAJ_DIR / f"{name}.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df["group"] = group
        dfs[name] = df
    else:
        print(f"[WARNING] Not found: {csv_path} — run src/run_all.py first")

# Load merged groups
ground_path  = TRAJ_DIR / "all_ground.csv"
balcony_path = TRAJ_DIR / "all_balcony.csv"

df_ground  = pd.read_csv(ground_path)  if ground_path.exists()  else None
df_balcony = pd.read_csv(balcony_path) if balcony_path.exists() else None

print(f"Individual recordings loaded : {list(dfs.keys())}")
print(f"Ground group  : {len(df_ground)} rows"  if df_ground  is not None else "Ground group  : not found")
print(f"Balcony group : {len(df_balcony)} rows" if df_balcony is not None else "Balcony group : not found")


## 3. Basic Statistics

### 3a. Per-recording summary

In [ ]:
rows = []
for name, _, group in RECORDINGS:
    if name not in dfs:
        continue
    df = dfs[name]
    if "speed_mps" not in df.columns:
        df = compute_speeds(df, fps=FPS, px_per_m=PX_PER_M[group])
    rows.append({
        "recording":    name,
        "group":        group,
        "n_ped":        df["id"].nunique(),
        "n_frames":     df["frame"].nunique(),
        "mean_speed":   round(df["speed_mps"].mean(), 3),
        "median_speed": round(df["speed_mps"].median(), 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


### 3b. Group-level statistics

In [ ]:
for label, df in [("Ground", df_ground), ("Balcony", df_balcony)]:
    if df is None:
        print(f"{label}: data not available")
        continue
    print(f'\n--- {label} group ---')
    print(f'  Pedestrians : {df["id"].nunique()}')
    print(f'  Rows        : {len(df)}')
    if "speed_mps" in df.columns:
        print(f'  Mean speed  : {df["speed_mps"].mean():.3f} m/s')
        print(f'  Median speed: {df["speed_mps"].median():.3f} m/s')


## 4. Visualisations

### 4a. Heatmap — balcony group (recommended for analysis)

In [ ]:
if df_balcony is not None:
    plot_heatmap(df_balcony, frame_w=1920, frame_h=1080,
                 output=str(RESULTS_DIR / "heatmap_balcony.png"))
else:
    print("Balcony data not available — skipping heatmap.")


### 4b. Trajectory overlay — per recording

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Pedestrian Trajectories per Recording", fontsize=14)
colors = cm.tab10.colors

for ax, (name, _, group) in zip(axes.flat, RECORDINGS):
    if name not in dfs:
        ax.set_title(f"{name} (no data)")
        ax.axis("off")
        continue
    df = dfs[name]
    for i, (pid, traj) in enumerate(df.groupby("id")):
        ax.plot(traj["x"], traj["y"], alpha=0.4, linewidth=0.8,
                color=colors[i % len(colors)])
    ax.set_title(f"{name}  [{group}]")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    ax.invert_yaxis()

plt.tight_layout()
out = RESULTS_DIR / "trajectories_per_recording.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved -> {out}")
plt.show()


### 4c. Speed distribution — ground vs balcony

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, df, color in [
        ("Ground (Records 1-2)",  df_ground,  "#e07b39"),
        ("Balcony (Records 3-6)", df_balcony, "#3b82f6")]:
    if df is not None and "speed_mps" in df.columns:
        ax.hist(df["speed_mps"].dropna(), bins=40, alpha=0.6,
                color=color, label=label, density=True)

ax.set_xlabel("Speed (m/s)")
ax.set_ylabel("Density")
ax.set_title("Speed distribution — Ground vs Balcony")
ax.legend()
fig.tight_layout()
out = RESULTS_DIR / "speed_ground_vs_balcony.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved -> {out}")
plt.show()


## 5. Load ETH/UCY Benchmark Data

In [ ]:
eth_df = load_eth_ucy(str(ETH_DIR), fps=2.5)
eth_stats = per_pedestrian_stats(eth_df, label="ETH/UCY")
print(f'ETH/UCY: {eth_df["id"].nunique()} pedestrians across {eth_df["dataset"].nunique()} scenes')
eth_df.head()


## 6. Comparison: Balcony group vs ETH/UCY

> Using the **balcony recordings** (3-6) for comparison as they share the overhead perspective of ETH/UCY.

In [ ]:
if df_balcony is not None:
    # Convert pixel coords to metres for fair comparison
    df_bal_m = df_balcony.copy()
    df_bal_m["x"] = df_bal_m["x"] / PX_PER_M["balcony"]
    df_bal_m["y"] = df_bal_m["y"] / PX_PER_M["balcony"]
    balcony_stats = per_pedestrian_stats(df_bal_m, label="Yaounde (balcony)")
    plot_comparison(balcony_stats, eth_stats, output_dir=str(RESULTS_DIR))
else:
    print("Balcony data not available.")


## 7. Per-recording Breakdown

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle("Speed distribution per recording vs ETH/UCY", fontsize=13)

eth_speeds = eth_df["speed_mps"].dropna()

for ax, (name, _, group) in zip(axes.flat, RECORDINGS):
    if name not in dfs:
        ax.set_title(f"{name} (no data)")
        ax.axis("off")
        continue
    df = dfs[name]
    if "speed_mps" not in df.columns:
        df = compute_speeds(df, fps=FPS, px_per_m=PX_PER_M[group])
    ax.hist(df["speed_mps"].dropna(), bins=30, alpha=0.7,
            color="#e07b39" if group == "ground" else "#3b82f6",
            label=f"{name} [{group}]", density=True)
    ax.hist(eth_speeds, bins=30, alpha=0.4, color="gray",
            label="ETH/UCY", density=True)
    ax.set_title(f"{name}  [{group}]")
    ax.set_xlabel("Speed (m/s)")
    ax.legend(fontsize=7)

plt.tight_layout()
out = RESULTS_DIR / "speed_per_recording.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved -> {out}")
plt.show()
